<a href="https://colab.research.google.com/github/yasyamauchi/education/blob/main/m5stack_imu2026F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 生体医工学実験(車椅子)用 m5stackのIMUデータ処理

2026/9/30　山内 更新履歴  
  
  2026年春学期のものを以下のように改造  
  * データファイルはGitHubから読むようにした（ドラッグアンドドロップ不要）
  * 介助ありなしを同時に処理
  
  2025年のものを以下のように改造  
  * A,Bのみとする
  * Xのみとする  
    
  2023年のものを以下のように改造  
  * サンプリング周波数は40Hzに上げる
  * X(前後方向)とY(左右方向)の回転角を扱う
  * スペクトログラムとフーリエ変換を行う

### Scipyの参考：  https://atatat.hatenablog.com/entry/data_proc_python4

# 1)m5Stackで得られたファイル"A.csv", "B.csv"をGoogle Colabにアップロードする  
ファイルはGitHubにあるので，班とグループを指定する．

In [ ]:
#@markdown ###班
han = 'A' #@param ["A","B","C","D","E","F"] {allow-input: false}
#@markdown ###グループ
group = '1' #@param ["1","2","3"] {allow-input: false}
print(han+"班"+group+"グループですね？間違っていたらもう一度選んでください．")

班とグループ名が正しければ次を実行する

In [ ]:
# @title
passA = "https://github.com/yasyamauchi/education/raw/refs/heads/main/data/bmeexp/IMU/"+han+"-"+group+"/A.csv"
passB = "https://github.com/yasyamauchi/education/raw/refs/heads/main/data/bmeexp/IMU/"+han+"-"+group+"/B.csv"
print(passA)
print(passB)
!wget "$passA"
!wget "$passB"

**←下図のように，「ファイル一覧」にA.csvとB.csvが出現したことを確認したら次へ**  

![Colab-filelist.png](https://github.com/yasyamauchi/education/blob/main/img/Colab-filelist.png?raw=1)

# 2)下準備

* サンプリング周波数は40[Hz]  
* 通過域端周波数は0.08[Hz]
* 阻止域端周波数は0.2[Hz]
* 通過域端最大損失は3[dB]
* 阻止域端最小損失は40[dB]

In [ ]:
# @title
# ライブラリのインポート
import numpy as np
from scipy import signal
from matplotlib import pyplot as plt

# ローパスフィルタの関数の定義
def lowpass(x, samplerate, fp, fs, gpass, gstop):
    fn = samplerate / 2   #ナイキスト周波数
    wp = fp / fn  #ナイキスト周波数で通過域端周波数を正規化
    ws = fs / fn  #ナイキスト周波数で阻止域端周波数を正規化
    N, Wn = signal.buttord(wp, ws, gpass, gstop)  #オーダーとバターワースの正規化周波数を計算
    b, a = signal.butter(N, Wn, "low")            #フィルタ伝達関数の分子と分母を計算
    y = signal.filtfilt(b, a, x)                  #信号に対してフィルタをかける
    return y

# その他
samplerate = 40 # サンプリング周波数は40Hz
fp = 0.08 # 通過域端周波数[Hz]
fs = 0.2 # 阻止域端周波数[Hz]
gpass = 3 # 通過域端最大損失[dB]
gstop = 40 # 阻止域端最小損失[dB]

# データの読み込み
m5stackA = np.loadtxt("A.csv",delimiter=',',skiprows=1)
m5stackB = np.loadtxt("B.csv",delimiter=',',skiprows=1)

# 3)計算と結果観察

## 結果１：生データを観察する  
RotX(前後の傾き)の時間変化  
赤線はローパスフィルター処理後

### 介助なし(A.csv)

In [ ]:
# @title

x = np.arange(0, m5stackA.shape[0]) / samplerate
RotXA = m5stackA[:,7]

# ローパスをする関数を実行
RotX_lofilt = lowpass(RotXA, samplerate, fp, fs, gpass, gstop)

# グラフ描画
fig = plt.figure(figsize=(16,8))

ax1 = fig.add_subplot(1, 1, 1)

ax1.plot(x,RotXA)
ax1.plot(x,RotX_lofilt,'r')
ax1.legend(['Raw','Lowpass'])
ax1.set_title('X axis Rotation', y=-0.2)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Rot. (deg)')
ax1.set_ylim(-10,10)
ax1.minorticks_on()
ax1.grid(color="black", linestyle="--")
fig.tight_layout(h_pad=4)              #レイアウトの設定
plt.show()

### 介助あり(B.csv)

In [ ]:
# @title

x = np.arange(0, m5stackB.shape[0]) / samplerate
RotXB = m5stackB[:,7]

# ローパスをする関数を実行
RotX_lofilt = lowpass(RotXB, samplerate, fp, fs, gpass, gstop)

# グラフ描画
fig = plt.figure(figsize=(16,8))

ax1 = fig.add_subplot(1, 1, 1)

ax1.plot(x,RotXB)
ax1.plot(x,RotX_lofilt,'r')
ax1.legend(['Raw','Lowpass'])
ax1.set_title('X axis Rotation', y=-0.2)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Rot. (deg)')
ax1.set_ylim(-10,10)
ax1.minorticks_on()
ax1.grid(color="black", linestyle="--")
fig.tight_layout(h_pad=4)              #レイアウトの設定
plt.show()

## 結果２：フーリエ変換を実行する  

### 介助なし(A.csv)

In [ ]:
# @title
from scipy.fft import fft, fftfreq

N = len(RotXA) # Length of the AccY data
yf1 = fft(RotXA) # Perform the Fourier Transform on RotX
xf1 = fftfreq(N, 1 / samplerate) # Generate frequency axis for the transformed data

# Plot the single-sided amplitude spectrum
fig = plt.figure(figsize=(10, 6))
plt.plot(xf1[0:N//2], np.abs(yf1[0:N//2])) # Plot the positive frequencies
fig.suptitle('FFT of X axis Rotation', y=0.0)
plt.xlabel('Frequency [Hz]')
plt.ylabel('Amplitude')
plt.ylim(0,8000)
plt.grid()

### 介助あり(B.csv)

In [ ]:
# @title
from scipy.fft import fft, fftfreq

N = len(RotXB) # Length of the AccY data
yf1 = fft(RotXB) # Perform the Fourier Transform on RotX
xf1 = fftfreq(N, 1 / samplerate) # Generate frequency axis for the transformed data

# Plot the single-sided amplitude spectrum
fig = plt.figure(figsize=(10, 6))
plt.plot(xf1[0:N//2], np.abs(yf1[0:N//2])) # Plot the positive frequencies
fig.suptitle('FFT of X axis Rotation', y=0.0)
plt.xlabel('Frequency [Hz]')
plt.ylabel('Amplitude')
plt.ylim(0,8000)
plt.grid()

## 結果３：スペクトログラムを観察する  
条件：窓関数サイズ512，ホップサイズ128

### 介助なし(A.csv)

In [ ]:
# @title
# 参考：https://nettyukobo.com/python_scipy_signal_spectrogram/

# 窓関数作成
win = signal.windows.hann(512)  # 窓関数

# スぺクトログラム作成
SFT = signal.ShortTimeFFT(win, hop=128, fs=samplerate, fft_mode='onesided2X', scale_to='magnitude')
Sxx = SFT.spectrogram(RotXA)
Sxx_dB = 10 * np.log10(np.fmax(Sxx, 1e-8))

# 時間軸と周波数軸の作成
N_t = np.arange(Sxx.shape[1])
t = SFT.delta_t * N_t
f = SFT.f

# グラフ作成
fig = plt.figure(figsize=(12,6))
fig.suptitle('Spectrogram of X axis Rotation', y=0.0)
plt.pcolormesh(t, f, Sxx_dB, cmap="jet")
plt.colorbar(label="Power [dbFS]")
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')

### 介助あり(B.csv)

In [ ]:
# @title
# 参考：https://nettyukobo.com/python_scipy_signal_spectrogram/

# 窓関数作成
win = signal.windows.hann(512)  # 窓関数

# スぺクトログラム作成
SFT = signal.ShortTimeFFT(win, hop=128, fs=samplerate, fft_mode='onesided2X', scale_to='magnitude')
Sxx = SFT.spectrogram(RotXB)
Sxx_dB = 10 * np.log10(np.fmax(Sxx, 1e-8))

# 時間軸と周波数軸の作成
N_t = np.arange(Sxx.shape[1])
t = SFT.delta_t * N_t
f = SFT.f

# グラフ作成
fig = plt.figure(figsize=(12,6))
fig.suptitle('Spectrogram of X axis Rotation', y=0.0)
plt.pcolormesh(t, f, Sxx_dB, cmap="jet")
plt.colorbar(label="Power [dbFS]")
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')

# 4)図を保存  
Windowsの場合，グラフの上で右クリックして「名前をつけて画像を保存」  
区別がつくように気をつける！